# 03 ? Encoder professor de skeleton (Hugging Face)

## Registro completo do check de compatibilidade

- Hub model: `mateuswiteck/tsfmxai-canonicalization-encoder-decoder`; variant `transformer_skeleton`; revision `2fcdcc88b41125bcfd92953c0bb8a668a603d167`.
- Configura??o publicada: skeleton-only (`predict_coefficients=false`), dimens?o 96, max_nodes 20, um canal, lags 1?16, grau 4, vocabulary 37. O target requerido para seis canais com mesmo range/grau precisa de 197 IDs.
- Seed/sample: `CombinatorialTreeSampler`, seed 20261125, sample index 4. O encoder ? congelado e o target `root` usa `prefix_values=0`, para n?o transferir valores de coefficients. Seis targets independentes foram gerados com shape `[6,96]` ap?s relabel local de fontes pr?prias para s?rie 0.
- Verifica??es: carregamento HF por commit/variant; target finito de shape `[1,96]`; perturba??o de coefficients sem m?scara mudou o vetor por L2 0.013765; com valores mascarados, mesma estrutura produziu target id?ntico.
- Ambiente: Python 3.11.9, PyTorch 2.12.0+cpu, CPU; CUDA indispon?vel.
- Conclus?o: encoder single-series pode supervisionar seis equa??es somente quando cada canal usa exclusivamente a pr?pria s?rie (ap?s relabel local para s?rie 0). Depend?ncias cruzadas continuam incompat?veis; o smoke abaixo verifica o caso independente. Sem treino de Chronos ou avalia??o cient?fica neste notebook.
- Reprodu??o: `python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/03_canonical_teacher_skeleton.ipynb`.

## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/03_canonical_teacher_skeleton.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.


In [1]:
from pathlib import Path
from dataclasses import replace
import json, sys
import torch
from huggingface_hub import hf_hub_download
ROOT=Path.cwd()
while not (ROOT/'src/tsfmxai').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/'src/tsfmxai').exists(): raise RuntimeError('repository root not found')
sys.path.insert(0,str(ROOT/'src'))
from tsfmxai.symbolic_regression.canonical_models import CanonicalEncoder, collate_canonical
from tsfmxai.symbolic_regression.canonical_data import CanonicalCodec
from tsfmxai.generators.combinatorial_expression import CombinatorialTreeConfig, CombinatorialTreeSampler
HF_REPO_ID='mateuswiteck/tsfmxai-canonicalization-encoder-decoder'
HF_REVISION='2fcdcc88b41125bcfd92953c0bb8a668a603d167'
ENCODER_VARIANT='transformer_skeleton'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
config_path=hf_hub_download(HF_REPO_ID,f'{ENCODER_VARIANT}/config.json',revision=HF_REVISION)
teacher_config=json.loads(Path(config_path).read_text(encoding='utf-8'))
encoder=CanonicalEncoder.from_pretrained((HF_REPO_ID,ENCODER_VARIANT),revision=HF_REVISION).to(DEVICE).eval()
assert teacher_config['model_kwargs']['predict_coefficients'] is False
assert all(not p.requires_grad for p in encoder.parameters())
print({'repo':HF_REPO_ID,'revision':HF_REVISION,'variant':ENCODER_VARIANT,'device':DEVICE,
       'dimension':encoder.dimension,'max_nodes':encoder.max_nodes,
       'predict_coefficients':teacher_config['model_kwargs']['predict_coefficients']})
from tsfmxai.generators.multiseries import MultiseriesConfig, NoiseConfig, sample_multiseries_system
from tsfmxai.representations.expression import Add, Multiply, Tanh, Lag


C:\Users\mateu\repos\tsfmxai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'repo': 'mateuswiteck/tsfmxai-canonicalization-encoder-decoder', 'revision': '2fcdcc88b41125bcfd92953c0bb8a668a603d167', 'variant': 'transformer_skeleton', 'device': 'cpu', 'dimension': 96, 'max_nodes': 20, 'predict_coefficients': False}


## Compatibilidade de vocabul?rio

O ID de cada lag inclui o ?ndice da s?rie. Logo, um vocabul?rio treinado para 1 s?rie n?o pode codificar fielmente equa??es com fontes nas s?ries 0?5. N?o vamos remapear IDs nem fingir compatibilidade.

In [2]:
from tsfmxai.datasets.multiseries import TrainingFormat
TEACHER_NUM_SERIES=teacher_config['codec']['num_series']
TEACHER_MIN_LAG=teacher_config['codec']['minimum_lag']
TEACHER_MAX_LAG=teacher_config['codec']['maximum_lag']
TEACHER_MAX_DEGREE=teacher_config['codec']['max_degree']
TEACHER_MAX_NODES=encoder.max_nodes
teacher_codec=CanonicalCodec(TEACHER_NUM_SERIES,TEACHER_MIN_LAG,TEACHER_MAX_LAG,TEACHER_MAX_DEGREE)
required_codec=CanonicalCodec(6,TEACHER_MIN_LAG,TEACHER_MAX_LAG,TEACHER_MAX_DEGREE)
assert teacher_codec.vocab_size==encoder.source_embedding.num_embeddings
compatible_six_series=(required_codec.vocab_size==encoder.source_embedding.num_embeddings)
print({'teacher_codec':{'series':TEACHER_NUM_SERIES,'lags':[TEACHER_MIN_LAG,TEACHER_MAX_LAG],
                        'degree':TEACHER_MAX_DEGREE,'vocab':teacher_codec.vocab_size},
       'six_series_vocab_required':required_codec.vocab_size,'six_series_compatible':compatible_six_series})
assert not compatible_six_series, 'Update this gate only after a six-series teacher is published.' 

# A one-series teacher can represent each equation only if it has no cross-series sources.

{'teacher_codec': {'series': 1, 'lags': [1, 16], 'degree': 4, 'vocab': 37}, 'six_series_vocab_required': 197, 'six_series_compatible': False}


## Vetor do encoder e sensibilidade a par?metros

O alvo ? o `root` do encoder, sem rodar decoder nem coefficient head. Embora a variante seja skeleton-only, a arquitetura de entrada cont?m valores num?ricos de par?metro. O teste abaixo mant?m a AST e altera somente os valores para verificar se o vetor muda.

In [3]:
sampler=CombinatorialTreeSampler(CombinatorialTreeConfig(
 minimum_operators=1,maximum_operators=2,num_series=1,minimum_lag=1,maximum_lag=16,max_degree=4))
example=teacher_codec.encode(sampler.sample(seed=20261125,sample_index=4))
batch=collate_canonical([example])
batch={k:v.to(DEVICE) for k,v in batch.items()}
changed={k:v.clone() for k,v in batch.items()}
parameter_nodes=changed['prefix_ids'].eq(3)
assert parameter_nodes.any(), 'sample must include a parameter leaf'
changed['prefix_values'][parameter_nodes]*=2.0
with torch.inference_mode():
 _,_,root=encoder(batch)
 _,_,changed_root=encoder(changed)
# The released encoder consumes numeric values, so define skeleton targets with those values masked.
masked={k:v.clone() for k,v in batch.items()}
changed_masked={k:v.clone() for k,v in changed.items()}
masked['prefix_values'].zero_(); changed_masked['prefix_values'].zero_()
with torch.inference_mode():
 _,_,teacher_target=encoder(masked)
 _,_,changed_teacher_target=encoder(changed_masked)
natural_delta=float(torch.linalg.vector_norm(root-changed_root).cpu())
masked_delta=float(torch.linalg.vector_norm(teacher_target-changed_teacher_target).cpu())
assert teacher_target.shape==(1,encoder.dimension) and torch.isfinite(teacher_target).all()
assert torch.equal(teacher_target,changed_teacher_target), 'masked skeleton target must ignore coefficient values'
print({'teacher_target_shape':tuple(teacher_target.shape),'finite':bool(torch.isfinite(teacher_target).all()),
       'unmasked_coefficient_perturbation_l2':natural_delta,
       'masked_target_change_under_same_skeleton':masked_delta,
       'teacher_frozen':all(not p.requires_grad for p in encoder.parameters())})

def localize_independent(node, channel):
 if isinstance(node,Lag):
  if node.variable != channel: raise ValueError('cross-series source cannot be mapped to the single-series teacher')
  return replace(node,variable=0)
 if isinstance(node,(Add,Multiply)): return type(node)(localize_independent(node.left,channel),localize_independent(node.right,channel))
 if isinstance(node,Tanh): return Tanh(localize_independent(node.operand,channel))
 return node

# Six channels, each with self-only mechanisms: preserve each per-channel skeleton exactly.
multiseries_config=MultiseriesConfig(
 ast_config=CombinatorialTreeConfig(minimum_operators=1,maximum_operators=2,minimum_lag=1,maximum_lag=16,max_degree=4),
 channel_count_probs={6:1.0},dependency_probs={'independent':1.0},treatment_probs={'original':1.0},
 max_external_sources=0,allow_no_lags=False,noise=__import__('tsfmxai.generators.multiseries',fromlist=['NoiseConfig']).NoiseConfig(std_range=(0.02,0.05)))
system=sample_multiseries_system(multiseries_config,seed=20261126)
assert all(tuple(sources)==(channel,) for channel,sources in enumerate(system.effective_sources))
localized=[replace(eq,expression=localize_independent(eq.expression,channel))
           for channel,eq in enumerate(system.equations)]
multibatch=collate_canonical([teacher_codec.encode(eq) for eq in localized])
multibatch={k:v.to(DEVICE) for k,v in multibatch.items()}
multibatch['prefix_values'].zero_()
with torch.inference_mode(): _,_,multi_teacher_targets=encoder(multibatch)
assert multi_teacher_targets.shape==(6,encoder.dimension) and torch.isfinite(multi_teacher_targets).all()
print({'independent_multiseries_teacher_targets':tuple(multi_teacher_targets.shape),
       'all_sources_self_only':True,'cross_series_topologies_supported':False})

{'teacher_target_shape': (1, 96), 'finite': True, 'unmasked_coefficient_perturbation_l2': 0.013765067793428898, 'masked_target_change_under_same_skeleton': 0.0, 'teacher_frozen': True}
{'independent_multiseries_teacher_targets': (6, 96), 'all_sources_self_only': True, 'cross_series_topologies_supported': False}


## Resultado, interpreta??o e gate

A variante correta do Hub foi carregada na revis?o imut?vel e produz embeddings finitos. O encoder sem coefficient head ainda recebe valores num?ricos: no input normal, alterar coefficients muda o embedding. Para construir um target que represente apenas o skeleton, esta execu??o zera `prefix_values` antes do encoder; o mesmo skeleton ent?o produz o mesmo target quando os coefficients mudam. Aplicar essa m?scara tamb?m em toda a gera??o/cache dos targets.

O teacher tem vocabul?rio de uma s?rie, mas pode supervisionar um batch de seis equa??es se cada equa??o usar somente sua pr?pria s?rie: relabeling local para ?ndice 0 preserva integralmente esse skeleton. O notebook verifica seis targets `[6,96]` nesse caso. **Limite cient?fico:** topologias com depend?ncias cruzadas entre canais n?o s?o represent?veis por este checkpoint. Para o experimento equation-head compat?vel, fixe `dependency_probs={'independent':1.0}` nos dois bra?os de compara??o; para rela??es cruzadas, publique um teacher treinado para seis s?ries.